# Predicting eddy tilt magnitude and direction from surface information

This workflow answers two different questions for **AE and CE separately**:

1. Can surface structure and environmental variables predict tilt in an unseen eddy?
2. Which feature groups have stable descriptive or predictive associations with tilt?

Tilt magnitude and circular direction are modelled separately. Model family and feature-set selection occur inside every outer eddy-grouped fold, so no repeatedly inspected final set is called untouched. Spatially blocked validation is added as a stress test. Feature ablation and beta-association results are evidence of association, not causality.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import seacofs_tilt_tools as tilt
import ml_subsurface_tools as ml

pd.set_option("display.max_columns", 50)
plt.rcParams["figure.dpi"] = 120

RANDOM_STATE = 42
OUTER_FOLDS = 5
INNER_FOLDS = 3
DIRECTION_MIN_TILT_KM = 5.0
GRID_ROTATION_DEG = 20.0

# Restrict the nested search to interpretable group comparisons. More
# detailed alternatives are assessed later as descriptive sensitivity tests.
NESTED_FEATURE_SETS = {
    name: ml.FEATURE_SETS[name]
    for name in ["full", "without_PV", "without_propagation",
                 "without_beta", "structure_environment", "beta_only"]
}

## 1. Load a common, fully observed vertical interval

Tilt is capped near 859 m, so only eddy-days whose vertical profile extends beyond 850 m are retained. PV-gradient terms must be based only on information that would be available at prediction time; verify that `add_pv_gradient_terms(..., core_mean=True)` does not use the subsurface tilt target or future information.

In [ ]:
paths = tilt.Paths()
grid = tilt.load_grid(paths.grid, paths.z_r)
df_eddies, df_tilt = tilt.load_tilt_tables(paths)
df_vert = tilt.load_vert(paths)

max_depth = (
    df_vert.groupby(["Eddy", "Day"], as_index=False)["Depth"]
    .max().rename(columns={"Depth": "max_profile_depth"})
)
df_eddies = df_eddies.merge(max_depth, on=["Eddy", "Day"], how="inner")
df_eddies = df_eddies[df_eddies["max_profile_depth"] > 850].copy()
df_eddies = tilt.add_pv_gradient_terms(df_eddies, grid, core_mean=True)
display(ml.target_availability_summary(df_eddies))

## 2. Engineer non-redundant predictors

The model uses `Rc` and `Omega` for surface structure; beta and bathymetry for environment; propagation; geographic ellipse orientation; and PV-gradient magnitude plus **unit direction components**. Raw PV east/north components are retained only for diagnostics because including them together with magnitude duplicates information.

`norm_time` is excluded because normalising by an eddy's final lifetime uses future knowledge. Propagation requires the previous observed position and therefore belongs to a trajectory-enabled prediction task, not a single-image task. The ellipse q-matrix is rotated +20 degrees from the model grid into geographic coordinates (equivalent to +40 degrees in doubled-angle space).

In [ ]:
model_df = ml.prepare_modelling_table(df_eddies, grid_rotation_deg=GRID_ROTATION_DEG)
polarity_data = {cyc: model_df[model_df["Cyc"] == cyc].copy() for cyc in ("AE", "CE")}

display(pd.Series(ml.FEATURES, name="model_feature").to_frame())
display(pd.DataFrame({
    cyc: {"rows": len(part), "eddies": part["Eddy"].nunique()}
    for cyc, part in polarity_data.items()
}).T)

ellipse_norm = np.sqrt(model_df["ellipse_major_sin2"] ** 2 + model_df["ellipse_major_cos2"] ** 2)
display(ellipse_norm.describe().rename("ellipse_encoding_norm"))

### Direct descriptive test of the polarity-specific PV hypothesis

For CE, the expected bearing is the raw PV gradient. For AE, it is the opposite bearing. This table tests the hypothesis directly before ML; it should not be inferred from model importance.

In [ ]:
hypothesis = []
for cyc, part in polarity_data.items():
    raw = ml.angular_error_deg(part["TiltDir"], part["PV_grad_theta"])
    expected = ml.angular_error_deg(part["TiltDir"], part["PV_reference_theta"])
    hypothesis.append({
        "Cyc": cyc, "median_error_raw_PV_deg": np.median(raw),
        "within_30deg_raw_PV": np.mean(raw <= 30),
        "median_error_expected_deg": np.median(expected),
        "within_30deg_expected": np.mean(expected <= 30),
    })
display(pd.DataFrame(hypothesis).set_index("Cyc").round(3))

## 3. Nested eddy-grouped prediction

Every outer fold holds out complete eddies. Within the remaining eddies, inner grouped CV selects Ridge versus restrained gradient boosting and selects a feature group. Thus each outer score represents a model-development process applied to unseen eddies. Each eddy receives equal total training weight.

Magnitude is modelled as `log1p(TiltDis)` and transformed back to kilometres. Direction is modelled as unit east/north components and is trained only where tilt is at least 5 km. This section is computationally intensive.

In [ ]:
nested = {}
for cyc, part in polarity_data.items():
    nested[cyc] = {}
    for task in ("magnitude", "direction"):
        task_data = part if task == "magnitude" else part[part["TiltDis"] >= DIRECTION_MIN_TILT_KM].copy()
        print(f"Nested CV: {cyc} {task} ({task_data['Eddy'].nunique()} eddies)")
        scores, selections, predictions = ml.nested_grouped_evaluation(
            task_data, task, feature_sets=NESTED_FEATURE_SETS,
            outer_splits=OUTER_FOLDS, inner_splits=INNER_FOLDS,
            minimum_tilt_km=DIRECTION_MIN_TILT_KM, random_state=RANDOM_STATE,
        )
        nested[cyc][task] = {"scores": scores, "selections": selections, "predictions": predictions}
        display(ml.summarise_outer_scores(scores, task).round(3))
        display(selections)

## 4. Out-of-fold calibration and direction diagnostics

All plotted predictions come from outer folds in which the relevant eddy was unseen. Look for systematic underprediction of large magnitudes and report direction separately at 5, 10 and 20 km thresholds.

In [ ]:
for cyc in ("AE", "CE"):
    magnitude_pred = nested[cyc]["magnitude"]["predictions"]
    direction_pred = nested[cyc]["direction"]["predictions"]
    ml.plot_magnitude_oof(magnitude_pred, title=f"{cyc}: nested out-of-fold magnitude")
    ml.plot_direction_oof(direction_pred, title=f"{cyc}: nested out-of-fold direction")
    display(ml.direction_performance_by_tilt(direction_pred, thresholds=(5, 10, 20)).round(3))

## 5. Full-data development sensitivity: feature groups

The nested outer results above are the unbiased prediction estimates. This section uses grouped CV on all available eddies to compare detailed feature removals. It is a development/association diagnostic, not a new test score. Differences should be compared with fold variability; tiny rank differences are not evidence that one representation is optimal.

In [ ]:
development = {}
for cyc, part in polarity_data.items():
    development[cyc] = {}
    for task in ("magnitude", "direction"):
        task_data = part if task == "magnitude" else part[part["TiltDis"] >= DIRECTION_MIN_TILT_KM].copy()
        config, cv_results = ml.select_configuration_grouped(
            task_data, task, feature_sets=ml.FEATURE_SETS, n_splits=INNER_FOLDS, random_state=RANDOM_STATE
        )
        development[cyc][task] = {"config": config, "cv_results": cv_results}
        print(cyc, task, config)
        display(ml.feature_set_comparison(cv_results, task).round(3))

## 6. Companion beta–tilt-magnitude analysis

Permutation or ablation importance does not show the direction or shape of an association. Here each eddy contributes one median beta and one median tilt magnitude, avoiding thousands of daily rows being treated as independent. Spearman correlation and an eddy bootstrap confidence interval provide a transparent unadjusted association. The binned curve shows shape but is not causal; beta may still encode geography and covary with `Rc`, `Omega`, bathymetry or EAC regime.

In [ ]:
beta_results = []
for cyc, part in polarity_data.items():
    result, eddy_table = ml.beta_magnitude_association(part, n_boot=1000, random_state=RANDOM_STATE)
    beta_results.append({"Cyc": cyc, **result})
    ml.plot_beta_relationship(eddy_table, title=f"{cyc}: eddy-level beta and tilt magnitude")
display(pd.DataFrame(beta_results).set_index("Cyc").round(3))

## 7. Propagation-confounding diagnostic

The EAC and coastline can impose a shared southwestward pathway. The table separates each eddy's full-track mean propagation from day-to-day anomalies around that mean. If track means associate with tilt but within-track anomalies do not, propagation is more likely acting as a pathway/regional proxy. Full-track means use future information and are **diagnostic only**, never predictive inputs.

In [ ]:
for cyc, part in polarity_data.items():
    prop_summary, prop_diagnostic = ml.propagation_confounding_summary(part)
    print(cyc)
    display(prop_summary.round(3))

## 8. Spatial-block stress test

Random unseen eddies can occupy the same EAC pathway as training eddies. This stress test assigns each complete eddy to a block using its median track position and holds out blocks. It uses the consensus configuration from nested folds, rather than retuning against spatial-test blocks. A large deterioration suggests that beta, bathymetry, propagation or PV terms partly encode regional structure.

In [ ]:
spatial_results = {}
for cyc, part in polarity_data.items():
    spatial_results[cyc] = {}
    for task in ("magnitude", "direction"):
        task_data = part if task == "magnitude" else part[part["TiltDis"] >= DIRECTION_MIN_TILT_KM].copy()
        config = ml.consensus_configuration(nested[cyc][task]["selections"])
        spatial = ml.spatial_block_evaluation(
            task_data, task, config, n_splits=4, minimum_tilt_km=DIRECTION_MIN_TILT_KM
        )
        spatial_results[cyc][task] = spatial
        metric = "eddy_weighted_magnitude_MAE_km" if task == "magnitude" else "eddy_weighted_mean_angular_error_deg"
        print(cyc, task, config)
        display(spatial[[metric]].agg(["mean", "std"]).round(3))

## 9. Interpretation and reporting rules

For AE and CE separately, report:

- nested outer-fold magnitude MAE, RMSE, R² and bias versus the eddy-median baseline;
- nested direction error and within-30° fraction versus mean-direction, PV-reference and propagation baselines;
- how often model families and feature sets are selected across outer folds;
- calibration for large tilt magnitudes and direction skill above 5, 10 and 20 km;
- the magnitude of feature-group improvements relative to fold-to-fold variability;
- the eddy-level beta association with its bootstrap interval;
- degradation under spatial blocking and the propagation-confounding diagnostic.

Use 'predictively useful' for nested-CV improvements, 'associated with' for stable descriptive relationships, and reserve causal language for a design that identifies causal effects. If a future publication requires one final performance number, lock a new set of eddies only after this workflow and all feature decisions are frozen.